# load_openaire_researchproduct_instances

Prototipo del nodo `load_openaire_researchproduct_instances` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_instances(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_instances = df[['id', 'instances', *_EXTRACTED_META_COLS]].explode('instances').reset_index(drop=True)

    df_instances = pd.json_normalize(df_research_instances['instances'])
    df_research_instances = pd.concat(
        [df_research_instances[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True), df_instances.reset_index(drop=True)],
        axis=1,
    )

    df_research_instances = df_research_instances.explode('pids').reset_index(drop=True)

    df_research_instances = df_research_instances.explode('urls').reset_index(drop=True)

    df_pids = pd.json_normalize(df_research_instances['pids'])
    df_research_instances = df_research_instances.drop(columns=['pids'])

    df_research_instances = pd.concat([df_research_instances, df_pids], axis=1)

    df_research_alternateidentifiers = (
        df_research_instances[['id', 'alternateIdentifiers', *_EXTRACTED_META_COLS]]
        .dropna()
        .explode('alternateIdentifiers')
        .reset_index(drop=True)
    )
    df_alternateidentifiers = pd.json_normalize(df_research_alternateidentifiers['alternateIdentifiers'])
    df_research_alternateidentifiers = pd.concat(
        [
            df_research_alternateidentifiers[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True),
            df_alternateidentifiers.reset_index(drop=True),
        ],
        axis=1,
    )

    df_research_instances.drop(columns=['alternateIdentifiers'], inplace=True)

    df_research_instances = _add_openaire_loaded_metadata(df_research_instances)
    df_research_alternateidentifiers = _add_openaire_loaded_metadata(df_research_alternateidentifiers)

    return df_research_instances, df_research_alternateidentifiers


In [ ]:
df_research_instances, df_research_alternateidentifiers = load_openaire_researchproduct_instances(df_researchproduct_raw)


In [ ]:
pd.DataFrame([
    {'dataset': 'df_research_instances', 'rows': len(df_research_instances), 'columns': len(df_research_instances.columns)},
    {'dataset': 'df_research_alternateidentifiers', 'rows': len(df_research_alternateidentifiers), 'columns': len(df_research_alternateidentifiers.columns)},
])


In [ ]:
df_research_instances.head(2)
